In [2]:
from ollama import Client
from ollama._types import Options
from pynvml import *
import time

In [3]:
nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

# GPU model name
name = nvmlDeviceGetName(handle)
name = name.replace("NVIDIA GeForce ","")

# Memory info
mem_info = nvmlDeviceGetMemoryInfo(handle)
total_vram_gb = mem_info.total / (1024**3)

# GPU model identifier in the perf tables
gpu_model = f"{name} {total_vram_gb:.0f}GB"

print(f"Testing performance on the following GPU: {gpu_model}")

Testing performance on the following GPU: RTX 5090 32GB


In [4]:
def get_vram_used():
    info = nvmlDeviceGetMemoryInfo(handle)
    return info.used / 10**9

In [5]:
initial_vram = get_vram_used()
print(f"Initial VRAM used: {initial_vram:.3f}GB")

Initial VRAM used: 1.237GB


In [6]:
client = Client()

In [12]:
ollama_version=!ollama --version
ollama_version=ollama_version[0].split()[-1]
ollama_version

'0.20.0'

In [13]:
def load_model():
    initial_vram = get_vram_used()
    response = client.chat(model=model, messages=[{'role': 'user', 'content': 'say hello'}], options=Options(num_ctx=context_length, num_predict=1))
    return get_vram_used()-initial_vram, response.load_duration/10**9

In [14]:
def compute_prompt_tokens(prompt):
    response = client.chat(model=model, messages=[{'role': 'user', 'content': prompt}], options=Options(num_ctx=context_length, num_predict=100))
    return response.prompt_eval_count

In [15]:
def measure_perfs(prompt_length):
    response = client.chat(model=model, messages=[{'role': 'user', 'content': "Summarize: " + (base_text*(int(prompt_length/base_tokens)+1))}], options=Options(num_ctx=context_length, num_predict=100))
    return response.prompt_eval_count,response.prompt_eval_count/(response.prompt_eval_duration/10**9), response.eval_count,response.eval_count/(response.eval_duration/10**9)

In [16]:
def unload_model():
    client.chat(model=model, messages=[{'role': 'user', 'content': 'say goodbye'}], options=Options(num_ctx=context_length, num_predict=1),
    keep_alive=0)
    time.sleep(1)
    return get_vram_used()-initial_vram

In [17]:
context_sizes = [1024*p for p in range(1,8)] + [8192*p for p in range(1,17)] + [16384*p for p in range(9,17)] + [32768*p for p in range(9,17)] + [65536*p for p in range(9,17)]

In [32]:
ctx1k = context_sizes[0]
ctx2k = context_sizes[1]
ctx4k = context_sizes[3]
ctx8k = context_sizes[7]
ctx16k = context_sizes[8]
ctx32k = context_sizes[10]
ctx64k = context_sizes[14]
ctx128k = context_sizes[22]
ctx192k = context_sizes[26]
ctx256k = context_sizes[30]
ctx384k = context_sizes[34]
ctx512k = context_sizes[38]
ctx1M = context_sizes[46]

In [50]:
print(",".join([f"{int(context_sizes[ctx_index]/1024)}k" for ctx_index in range(7,35)]))
print(",".join([f"{context_sizes[ctx_index]}" for ctx_index in range(7,35)]))

8k,16k,24k,32k,40k,48k,56k,64k,72k,80k,88k,96k,104k,112k,120k,128k,144k,160k,176k,192k,208k,224k,240k,256k,288k,320k,352k,384k
8192,16384,24576,32768,40960,49152,57344,65536,73728,81920,90112,98304,106496,114688,122880,131072,147456,163840,180224,196608,212992,229376,245760,262144,294912,327680,360448,393216


In [19]:
base_text = '\nAbsolutely! Let’s unpack this carefully and thoroughly. The concept of a **token** comes up most often in natural language processing (NLP), programming, and blockchain/crypto, but I’ll focus first on the context most relevant to AI models like me, and then touch briefly on other uses for clarity.\n\n---\n\n## **1. Tokens in AI and NLP (Language Models)**\n\nIn the context of language models like GPT:\n\n**A token is essentially a piece of text that the model treats as a unit for processing.**\n\nThink of it as a “chunk” of text, which could be:\n\n* A word (“apple” → 1 token)\n* Part of a word (“unbelievable” → might be split into “un”, “believ”, “able” → 3 tokens)\n* A single character or punctuation mark ("," or "!" → 1 token)\n* Even whitespace in some systems.\n\nLanguage models **do not process text character by character or word by word** in the strict sense; they process these tokens. This is because using tokens allows the model to handle text efficiently while capturing meaningful patterns.\n\n### **Why not just words?**\n\nWords are tricky because:\n\n* Some words are very long and rare.\n* Some languages (like Chinese, Japanese) don’t use spaces to separate words.\n* Using subword tokens balances between vocabulary size and representation power.\n\nBy using tokens, a model can efficiently represent language while keeping the total vocabulary manageable.\n\n---\n\n### **Tokenization: How text becomes tokens**\n\nThe process of breaking text into tokens is called **tokenization**. There are different tokenization methods:\n\n1. **Whitespace tokenization**\n\n   * Splits text at spaces.\n   * `"I love pizza!"` → `["I", "love", "pizza!"]`\n   * Simple, but can miss nuances like punctuation.\n\n2. **Subword tokenization (most modern models use this)**\n\n   * Breaks rare words into smaller pieces.\n   * `"unbelievable"` → `["un", "believ", "able"]`\n   * Helps the model understand new words it hasn\'t seen during training.\n\n3. **Character-level tokenization**\n\n   * Treats each character as a token.\n   * `"cat"` → `["c", "a", "t"]`\n   * Rarely used alone because sequences become very long.\n\n4. **Byte Pair Encoding (BPE)**\n\n   * A popular method used in GPT models.\n   * Starts with characters and iteratively merges common sequences to form subwords.\n   * Efficiently handles rare words while keeping the vocabulary compact.\n\n---\n\n### **Example of token counts**\n\nLet’s take the sentence:\n\n```\nHello, world! I love programming.\n```\n\nA GPT-style tokenizer might break it into tokens like:\n\n```\n["Hello", ",", " world", "!", " I", " love", " program", "ming", "."]\n```\n\nHere, it’s **9 tokens**. Notice:\n\n* Punctuation is separate.\n* Spaces before words are included as part of the token in some tokenizers.\n* Words like “programming” are split.\n\n---\n\n### **Why tokens matter in language models**\n\n1. **Input length**\n\n   * Models have a limit on the number of tokens they can process at once (e.g., GPT-4’s 8,192 or 32k token limit).\n   * Longer text → more tokens → may exceed the model’s limit.\n\n2. **Cost & computation**\n\n   * In APIs like OpenAI’s, you pay per token.\n   * More tokens → more computation → higher cost.\n\n3. **Context & memory**\n\n   * Models can only “remember” information within a token limit.\n   * Efficient tokenization allows models to fit more meaningful text into memory.\n\n---\n\n## **2. Tokens in Programming**\n\nA token in programming is a **syntactic unit of code**:\n\n* Keywords (`if`, `for`, `return`)\n* Identifiers (`myVariable`, `functionName`)\n* Operators (`+`, `-`, `=`)\n* Literals (`42`, `"hello"`)\n* Punctuation (`;`, `{`, `}`)\n\n**Example:**\n\n```python\nx = 42 + 7\n```\n\nTokenization would produce:\n\n```\n["x", "=", "42", "+", "7"]\n```\n\nThis is similar in principle: breaking a stream of characters into meaningful chunks the compiler or interpreter can understand.\n\n---\n\n## **3. Tokens in Blockchain / Crypto**\n\nHere, a token is a **digital asset** issued on a blockchain, representing something of value:\n\n* **Utility tokens** – give access to a service (e.g., Binance Coin for exchange fees).\n* **Security tokens** – represent ownership or investment.\n* **NFTs (non-fungible tokens)** – unique digital items (art, collectibles).\n\nThis is different from NLP, but the common theme is that a token is a **unit of something meaningful**, whether text, code, or digital value.\n\n---\n\n## **Key Takeaways**\n\n* **In AI/NLP:** Token = smallest meaningful piece of text for a model to process.\n* **In programming:** Token = smallest meaningful element of code.\n* **In crypto:** Token = a digital representation of value or rights.\n\n**Analogy:** Tokens are like LEGO blocks. Each block may be a full piece (word) or a part of a bigger structure (subword). You can build complex things (sentences, programs, ecosystems) from these basic building blocks.\n\n---\n\nIf you want, I can make a **visual diagram showing how a sentence gets broken into tokens and why that matters for GPT**, which makes this concept much easier to grasp.\n\nDo you want me to do that?\n'

In [20]:
for model in client.list().models:
    print(f"{model.model},{model.details.quantization_level},{model.size/10**9:.3f}")

granite4:tiny-h,Q4_K_M,4.231
granite4:micro-h,Q4_K_M,1.943
qwen3.5:9b,Q4_K_M,6.594
lfm2.5-thinking:1.2b,Q4_K_M,0.731
qwen3.5:2b,Q8_0,2.741
qwen3.5:0.8b,Q8_0,1.036
glm-ocr:q8_0,Q8_0,1.588
gemma4:31b,Q4_K_M,19.869
gemma4:26b,Q4_K_M,17.988
gemma4:e4b,Q4_K_M,9.608
gemma4:e2b,Q4_K_M,7.162
qwen3.5:4b,Q4_K_M,3.390
embeddinggemma:300m,BF16,0.622
devstral-small-2:24b,Q4_K_M,15.177
qwen3.5:35b,Q4_K_M,23.869
gemma3:27b,Q4_K_M,17.397
qwen3.5:27b-130k,Q4_K_M,17.420
qwen3.5:27b,Q4_K_M,17.420
qwen3.5:35b-190k,Q4_K_M,23.869
qwen3.5:27b-q4_K_M,Q4_K_M,17.420
qwen3-coder-next:latest,Q4_K_M,51.742
qwen3-vl:30b,Q4_K_M,19.595
qwen3:32b,Q4_K_M,20.201
qwen3:30b,Q4_K_M,18.557
nemotron-3-nano:30b,Q4_K_M,24.272
qwen3-coder:30b,Q4_K_M,18.557
qwen3-vl:32b,Q4_K_M,20.910
glm-4.7-flash:q4_K_M,Q4_K_M,18.766
devstral-small-2:24b-100k,Q4_K_M,15.177
ministral-3:14b,Q4_K_M,9.083
qwen3-vl:8b,Q4_K_M,6.140
ministral-3:8b,Q4_K_M,6.022
qwen3-vl:4b,Q4_K_M,3.296
ministral-3:3b,Q4_K_M,2.954
qwen3-vl:2b,Q4_K_M,1.890
functiongemma:

In [47]:
models=[
("gemma4:31b",ctx256k),
("gemma4:26b",ctx256k),
("gemma4:e4b",ctx128k),
("gemma4:e2b",ctx128k),
("qwen3.5:27b",ctx256k),
("qwen3.5:35b",ctx256k),
("qwen3.5:9b",ctx256k),
("qwen3.5:4b",ctx256k),
("qwen3.5:2b",ctx256k),
("qwen3.5:0.8b",ctx256k),
("glm-4.7-flash:q4_K_M",ctx192k),
("gpt-oss:20b",ctx128k),
("devstral-small-2:24b",ctx384k),
("mistral-small3.2:24b",ctx128k),
("ministral-3:14b",ctx256k),
("ministral-3:8b",ctx256k),
("ministral-3:3b",ctx256k),
("lfm2.5-thinking:1.2b",ctx128k),
("granite4:small-h",ctx128k),
("granite4:tiny-h",ctx128k),
("granite4:micro-h",ctx128k),
("gemma3:27b",ctx128k),
("gemma3:4b",ctx128k),
("gemma3:12b",ctx128k),
("gemma3:1b",ctx32k),
("gemma3n:e4b",ctx32k),
("gemma3n:e2b",ctx32k),
("embeddinggemma:300m",ctx2k),
("qwen3-embedding:0.6b",ctx32k),
("nomic-embed-text:latest",ctx2k),
("glm-ocr:q8_0",ctx128k),
]

In [37]:
unload_model()
print(f"Model {model} unloaded from VRAM")

Model gemma4:e2b unloaded from VRAM


In [ ]:
for model,max_context_length in models[4:]:
    context_vram_list = []
    last_context_vram = 0
    for ctx_index in range(7,35):
        context_length = context_sizes[ctx_index]
        if context_length <= max_context_length and last_context_vram < 32:
            model_vram_gb, load_time_sec = load_model()
            print(f"Model {model} loaded in {load_time_sec:.2f} sec - {model_vram_gb:.3f}GB VRAM at {int(context_length/1024)}k context length")
            context_vram_list.append(f"{model_vram_gb:.3f}")
            last_context_vram = model_vram_gb
            unload_model()
        else:
            context_vram_list.append("")
    print(f",{load_time_sec:.2f},{max_context_length},{','.join(context_vram_list)}")

Model qwen3.5:27b loaded in 9.23 sec - 23.932GB VRAM at 8k context length
Model qwen3.5:27b loaded in 6.41 sec - 24.519GB VRAM at 16k context length
Model qwen3.5:27b loaded in 6.95 sec - 25.123GB VRAM at 24k context length
Model qwen3.5:27b loaded in 6.70 sec - 25.725GB VRAM at 32k context length
Model qwen3.5:27b loaded in 8.47 sec - 26.329GB VRAM at 40k context length
Model qwen3.5:27b loaded in 8.68 sec - 26.990GB VRAM at 48k context length
Model qwen3.5:27b loaded in 6.38 sec - 27.629GB VRAM at 56k context length
Model qwen3.5:27b loaded in 6.73 sec - 28.261GB VRAM at 64k context length
Model qwen3.5:27b loaded in 6.93 sec - 28.900GB VRAM at 72k context length
Model qwen3.5:27b loaded in 7.25 sec - 29.535GB VRAM at 80k context length
Model qwen3.5:27b loaded in 7.60 sec - 30.173GB VRAM at 88k context length
Model qwen3.5:27b loaded in 10.06 sec - 30.808GB VRAM at 96k context length
Model qwen3.5:27b loaded in 9.93 sec - 31.446GB VRAM at 104k context length
Model qwen3.5:27b loaded

In [44]:
model = "gemma4:31b"
max_context_length = ctx256k

In [150]:
base_tokens = compute_prompt_tokens(base_text)
base_tokens

1273

In [175]:
previous_repeat = -1
for prompt_size in context_sizes:
    if prompt_size > max_context_length:
        break
    repeat = int(prompt_size/base_tokens)
    if repeat != previous_repeat:
        start_time = time.time()
        prompt_tokens,prompt_tokens_per_sec, output_tokens,output_tokens_per_sec = measure_perfs(prompt_size)
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"{gpu_model},{model},{prompt_tokens},{int(prompt_tokens_per_sec)},{output_tokens_per_sec:.2f}")
        if elapsed_time > 60:
            break
    previous_repeat = repeat        

RTX 5090 32GB,gemma4:e2b,1278,3020,32.43
RTX 5090 32GB,gemma4:e2b,2537,4056,26.84
RTX 5090 32GB,gemma4:e2b,3796,3980,24.51
RTX 5090 32GB,gemma4:e2b,5055,3920,22.85
RTX 5090 32GB,gemma4:e2b,6314,3906,21.94
RTX 5090 32GB,gemma4:e2b,7573,3785,19.26
RTX 5090 32GB,gemma4:e2b,8832,3758,18.13
RTX 5090 32GB,gemma4:e2b,16386,547,12.75
RTX 5090 32GB,gemma4:e2b,25199,315,9.65
